# Ficheros · Avanzado

Análisis de textos, JSON y CSV reales. Los ficheros son los de la carpeta `data` de [30 Days Of Python](https://github.com/Asabeneh/30-Days-Of-Python) y se descargan al ejecutar el notebook.

## 1. Descargar los datos

Descargar los ficheros necesarios en la carpeta del notebook.

In [1]:
from urllib import request

URL_30_DIAS = "https://raw.githubusercontent.com/Asabeneh/30-Days-Of-Python/master/data/"
FICHEROS = ["obama_speech.txt", "michelle_obama_speech.txt", "donald_speech.txt", "melina_trump_speech.txt",
            "countries_data.json", "email_exchanges_big.txt", "romeo_and_juliet.txt", "hacker_news.csv", "stop_words.py"]
for fichero in FICHEROS:
    request.urlretrieve(URL_30_DIAS + fichero, fichero)
print("Descargados:", len(FICHEROS))

Descargados: 9


## 2. Líneas y palabras

Escribir una función que cuente las líneas y las palabras de un fichero de texto y aplicarla a los cuatro discursos.

In [2]:
def lineas_y_palabras(ruta):
    with open(ruta, encoding="utf-8") as f:
        texto = f.read()
    return len(texto.splitlines()), len(texto.split())

for discurso in FICHEROS[:4]:
    print(discurso, lineas_y_palabras(discurso))

obama_speech.txt (66, 2400)
michelle_obama_speech.txt (83, 2204)
donald_speech.txt (48, 1259)
melina_trump_speech.txt (33, 1375)


## 3. Datos de países en JSON

Con `countries_data.json`: contar los idiomas distintos, obtener los $n$ idiomas más hablados (en cuántos países se hablan), los $n$ países más poblados y ordenar los países por nombre, capital y población.

In [3]:
import json
from collections import Counter

with open("countries_data.json", encoding="utf-8") as f:
    paises = json.load(f)

def most_spoken_languages(n):
    return [(c, idioma) for idioma, c in Counter(i for p in paises for i in p["languages"]).most_common(n)]

def most_populated_countries(n):
    top = sorted(paises, key=lambda p: p["population"], reverse=True)[:n]
    return [{"country": p["name"], "population": p["population"]} for p in top]

print(len({i for p in paises for i in p["languages"]}))
print(most_spoken_languages(10))
print(most_populated_countries(3))
for clave in ("name", "capital", "population"):
    print(clave, [p["name"] for p in sorted(paises, key=lambda p: p[clave])[:3]])

112
[(91, 'English'), (45, 'French'), (25, 'Arabic'), (24, 'Spanish'), (9, 'Portuguese'), (9, 'Russian'), (8, 'Dutch'), (7, 'German'), (5, 'Chinese'), (4, 'Serbian')]
[{'country': 'China', 'population': 1377422166}, {'country': 'India', 'population': 1295210000}, {'country': 'United States of America', 'population': 323947000}]
name ['Afghanistan', 'Albania', 'Algeria']
capital ['Antarctica', 'Bouvet Island', 'United States Minor Outlying Islands']
population ['Bouvet Island', 'Heard Island and McDonald Islands', 'South Georgia and the South Sandwich Islands']


## 4. Correos entrantes

Extraer la lista de direcciones de correo de los remitentes (líneas `From ...`) de `email_exchanges_big.txt`.

In [4]:
import re

with open("email_exchanges_big.txt", encoding="utf-8") as f:
    correos = re.findall(r"^From (\S+@\S+)", f.read(), flags=re.MULTILINE)
print(len(correos), len(set(correos)), correos[:5])

1797 46 ['stephen.marquard@uct.ac.za', 'louis@media.berkeley.edu', 'zqian@umich.edu', 'rjlowe@iupui.edu', 'zqian@umich.edu']


## 5. Palabras más comunes

Escribir `find_most_common_words`, que reciba una cadena o un fichero y un número $n$, y devuelva las $n$ palabras más frecuentes en orden descendente. Aplicarla a los discursos y a *Romeo y Julieta*.

In [5]:
import os

def find_most_common_words(fuente, n):
    if os.path.isfile(fuente):
        with open(fuente, encoding="utf-8") as f:
            fuente = f.read()
    return [(c, p) for p, c in Counter(re.findall(r"[A-Za-z']+", fuente)).most_common(n)]

print(find_most_common_words("the cat and the hat and the bat", 2))
for fichero in FICHEROS[:4] + ["romeo_and_juliet.txt"]:
    print(fichero, find_most_common_words(fichero, 10))

[(3, 'the'), (2, 'and')]
obama_speech.txt [(120, 'the'), (107, 'and'), (81, 'of'), (66, 'to'), (58, 'our'), (50, 'we'), (49, 'that'), (48, 'a'), (36, 'is'), (23, 'us')]
michelle_obama_speech.txt [(84, 'to'), (80, 'and'), (78, 'the'), (46, 'of'), (41, 'a'), (40, 'that'), (36, 'in'), (27, 'for'), (27, 'our'), (26, 'my')]
donald_speech.txt [(61, 'the'), (53, 'and'), (40, 'will'), (38, 'of'), (32, 'to'), (30, 'our'), (27, 'we'), (20, 'is'), (17, 'America'), (16, 'We')]
melina_trump_speech.txt [(73, 'and'), (54, 'to'), (48, 'the'), (29, 'is'), (28, 'I'), (27, 'for'), (25, 'of'), (22, 'a'), (19, 'that'), (18, 'you')]
romeo_and_juliet.txt [(767, 'the'), (575, 'I'), (563, 'to'), (562, 'and'), (487, 'of'), (457, 'a'), (342, 'in'), (334, 'is'), (318, 'you'), (310, 'my')]


## 6. Similitud entre textos

Comparar dos discursos: limpiar el texto (`clean_text`), quitar las palabras vacías de `stop_words.py` (`remove_support_words`) y medir la similitud (`check_text_similarity`) como el índice de Jaccard de sus vocabularios.

In [6]:
import ast

with open("stop_words.py", encoding="utf-8") as f:
    texto = f.read()
stop_words = set(ast.literal_eval(texto[texto.index("["):].strip()))

def clean_text(texto):
    return re.sub(r"[^a-z\s]", " ", texto.lower())

def remove_support_words(texto):
    return {p for p in texto.split() if p not in stop_words}

def check_text_similarity(a, b):
    va, vb = (remove_support_words(clean_text(open(t, encoding="utf-8").read())) for t in (a, b))
    # |A ∩ B| / |A ∪ B|
    return len(va & vb) / len(va | vb)

print(f"{check_text_similarity('michelle_obama_speech.txt', 'melina_trump_speech.txt'):.2%}")
print(f"{check_text_similarity('obama_speech.txt', 'donald_speech.txt'):.2%}")

18.18%
13.87%


## 7. Hacker News

En `hacker_news.csv`, contar las líneas que contienen `python`/`Python`, las que contienen `JavaScript`/`javascript`/`Javascript` y las que contienen `Java` pero no `JavaScript`.

In [7]:
with open("hacker_news.csv", encoding="utf-8") as f:
    lineas = f.read().splitlines()[1:]

print(sum("python" in l.lower() for l in lineas))
print(sum("javascript" in l.lower() for l in lineas))
print(sum(re.search(r"java(?!script)", l, re.IGNORECASE) is not None for l in lineas))

179
184
72
